In [ ]:
import pandas as pd

def print_validation_report(df: pd.DataFrame):
    """Generates a comprehensive summary of data health."""
    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)

    # 1. Shape & Duplicates
    print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    dup_count = df.duplicated(subset=["event_id"]).sum()
    print(f"Duplicate event_id rows: {dup_count}")

    # 2. Data Types
    print("\n── Data Types ──────────────────────────────────────────────")
    print(df.dtypes.value_counts().to_string())

    # 3. Missing Values
    print("\n── Missing Values ──────────────────────────────────────────")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"count": missing, "pct (%)": missing_pct})
    missing_df = missing_df[missing_df["count"] > 0].sort_values(by="pct (%)", ascending=False)
    
    if missing_df.empty:
        print("No missing values found.")
    else:
        print(missing_df.to_string())
    print("\n" + "=" * 60)
df_final = pd.read_csv("./data/intermi/final_chess_dataset.csv")
# Run the report on our merged dataset
print_validation_report(df_final)

In [ ]:
# Create the target variables based on the 'result' column
# Mapping logic based on standard chess notation (1-0: White, 0-1: Black, 1/2-1/2: Draw)
# Lichess data string formats ("White", "Black", "Draw") were harmonized in Task 1.

multiclass_map = {"1-0": 2, "White": 2, "1/2-1/2": 1, "Draw": 1, "0-1": 0, "Black": 0}
binary_map = {"1-0": 1.0, "White": 1.0, "0-1": 0.0, "Black": 0.0}

df_final["winner_multiclass"] = df_final["result"].map(multiclass_map)
df_final["winner_binary"] = df_final["result"].map(binary_map)

print("── Target: winner_multiclass ───────────────────────────────")
label_map = {0: "Black wins", 1: "Draw", 2: "White wins"}
print(df_final["winner_multiclass"].value_counts().rename(label_map).to_string())

print("\n── Target: winner_binary ───────────────────────────────────")
print(df_final["winner_binary"].value_counts().rename({0.0: "Black wins", 1.0: "White wins"}).to_string())
print(f"Excluded (Draws/NaN): {df_final['winner_binary'].isna().sum():,}")